# Location Attributes

Generate the `location_attributes.csv` file for the backoffice upload.

**Input:**
- `data/raw/Mangrove_Breakthrough_Countries_20260528.xlsx` — list of countries that have committed to the Mangrove Breakthrough (shared by email, uploaded to bucket)
- `data/raw/locations_merged.csv` — platform locations with UUIDs

**Output:**
- `data/processed/location_attributes.csv` — CSV with columns `location_id`, `legal_status`, `mangrove_breakthrough_committed`

**Notes:**
- Uploading this file to the backoffice **replaces all** existing location attribute records.
- `legal_status` is left empty for now (data not yet available).
- South Korea (listed in the Excel) has no matching entry in `locations_merged.csv` (neither ISO `KOR` nor `PRK` exist). It is excluded from the output.

In [ ]:
import csv
from pathlib import Path

import openpyxl

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

EXCEL_PATH = RAW / "Mangrove_Breakthrough_Countries_20260528.xlsx"
LOCATIONS_PATH = RAW / "locations_merged.csv"
OUTPUT_PATH = PROCESSED / "location_attributes.csv"

## 1. Read Mangrove Breakthrough countries from Excel

In [ ]:
wb = openpyxl.load_workbook(EXCEL_PATH, read_only=True)
ws = wb.active

breakthrough_countries = []
for i, row in enumerate(ws.iter_rows(values_only=True)):
    if i == 0:
        continue  # skip header
    breakthrough_countries.append(row[0].strip())

print(f"{len(breakthrough_countries)} Mangrove Breakthrough countries:")
for c in breakthrough_countries:
    print(f"  - {c}")

## 2. Load platform locations and build country lookup

In [ ]:
with open(LOCATIONS_PATH) as f:
    reader = csv.DictReader(f)
    country_to_id = {
        row["name"]: row["id"]
        for row in reader
        if row["type"] == "country"
    }

print(f"{len(country_to_id)} countries in locations_merged.csv")

## 3. Match breakthrough countries to location IDs

Name mapping is needed for "Mexico" (listed as "Mexico" in Excel, stored as "México" in the platform).

South Korea is listed in the Mangrove Breakthrough Excel but does **not** exist in `locations_merged.csv`. It is excluded from the output.

In [ ]:
# Name corrections: Excel name -> platform name
NAME_MAP = {
    "Mexico": "México",
}

breakthrough_names = set()
unmatched = []

for country in breakthrough_countries:
    name = NAME_MAP.get(country, country)
    if name in country_to_id:
        breakthrough_names.add(name)
    else:
        unmatched.append(country)

print(f"Matched: {len(breakthrough_names)}/{len(breakthrough_countries)}")
if unmatched:
    print(f"Unmatched (excluded from output): {unmatched}")

## 4. Write location_attributes.csv

All 122 countries are included. Breakthrough countries get `true`, the rest get `false`. This is necessary because uploading replaces all existing location attribute records.

In [ ]:
with open(OUTPUT_PATH, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["location_id", "legal_status", "mangrove_breakthrough_committed"])
    for name, loc_id in sorted(country_to_id.items(), key=lambda x: int(x[1])):
        committed = "true" if name in breakthrough_names else "false"
        writer.writerow([loc_id, "", committed])

print(f"Written {len(country_to_id)} rows to {OUTPUT_PATH}")
print(f"  - {len(breakthrough_names)} with mangrove_breakthrough_committed=true")
print(f"  - {len(country_to_id) - len(breakthrough_names)} with mangrove_breakthrough_committed=false")